# Imports

In [43]:
import torch
import sys
import os
import joblib
import warnings
from torch import Tensor
sys.path.append('../..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objs as go
from definitions import DATA_DIR, EXPERIMENTS_DIR, ROOT_DIR
from src.training.hierarchecal_articles_classification import load_data,get_configs,Config,ArticleAsKG
from src import utils
from umap import UMAP
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
    roc_curve
)
from PIL import ImageColor
from sklearn.metrics._classification import UndefinedMetricWarning
from sklearn.metrics import multilabel_confusion_matrix
from torch_scatter import scatter

In [2]:
warnings.filterwarnings(action = 'ignore', category = UndefinedMetricWarning)

In [3]:
px.defaults.template = 'seaborn'

In [4]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [5]:
TASK = 'hierarchical_classification'
EXPERIMENT = 'sub_dsis'
CORPUS = 'large'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, TASK, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORY_PATH = os.path.join(EXPERIMENT_DIR, 'history.csv')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [6]:
history = pd.read_csv(HISTORY_PATH)
history = history \
    .drop(columns = ['iteration']) \
    .groupby(['epoch','phase','exp','trial']) \
    .mean() \
    .reset_index()
history.rename(columns = { "exp" : "expr" }, inplace = True)
history.head()

,epoch,phase,expr,trial,loss,accuracy,f1_score
0,0,train,Attn_Attn_null_False,0,0.224554,0.915003,0.229180
1,0,train,Attn_Attn_null_False,1,0.226996,0.910632,0.218136
2,0,train,Attn_Attn_null_False,2,0.224316,0.914520,0.218658
3,0,train,Attn_Attn_null_False,3,0.256363,0.894919,0.171703
4,0,train,Attn_Attn_null_True,0,0.308878,0.874604,0.875767


In [7]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','expr']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','expr']).std() \
    .add_suffix('_std')

In [8]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,expr,loss_mean,accuracy_mean,f1_score_mean,loss_std,accuracy_std,f1_score_std
0,0,train,Attn_Attn_null_False,0.233057,0.908768,0.209419,0.015584,0.009438,0.025654
1,0,train,Attn_Attn_null_True,0.304913,0.876047,0.879332,0.009486,0.006817,0.007880
2,0,train,Attn_Max_null_False,0.221938,0.912846,0.218918,0.026709,0.014519,0.006198
3,0,train,Attn_Max_null_True,0.306985,0.876044,0.876856,0.026076,0.016851,0.018613
4,0,train,Attn_Mean_null_False,0.240984,0.903805,0.237648,0.007756,0.003531,0.008795


# Load Data

In [9]:
base_config = utils.load_dataclass(Config, CONFIG_PATH)
configs = get_configs(base_config)

In [10]:
data = load_data(next(iter(configs.values())), splits_path = SPLIT_PATH)

Labled Articles : 7108 / 7343
Number of articles : 7343
Number of edges between articles : 884968
Number of entities : 47064
Number of edges (triples) between entities : 68773
Number of edges between articles and triples : 72315
Number of edges between articles and entities : 97608 


Loading prexisting split.



In [35]:
metadata = joblib.load(filename = os.path.join(DATA_DIR, CORPUS, 'data.articles.pkl'))

# Learning Curves

In [11]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['expr'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and expr == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [12]:
line_plot(history, 'loss')

In [13]:
line_plot(history, 'accuracy')

In [14]:
line_plot(history, 'f1_score')

# Calculate other metrics

## Load Predictions

In [15]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [16]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [17]:
pkl_data = joblib.load(filename = os.path.join(DATA_DIR, CORPUS, "data.articles.pkl"))

In [18]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [19]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [20]:
def get_label_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    for label in range(data.y.size(-1)):
            
        for (expr,trial),result in results.items():

            y = data.y[data.split == int(trial)]
            y_hat = result[data.split == int(trial)].sigmoid()
            y_hard = y_hat.gt(0.5).long()
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label]),
                "precision" : precision_score(y[:,label], y_hard[:,label]),
                "recall" : recall_score(y[:,label], y_hard[:,label]),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [21]:
def get_row_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data.y[data.split == int(trial)]
        y_hat = result[data.split == int(trial)].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro'),
            "precision" : precision_score(y, y_hard , average='macro'),
            "recall" : recall_score(y, y_hard , average='macro'),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [22]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics(data.articles, results, pkl_data['teams'])
display_mean_std_df(label_wise_metrics_agg)

,expr,recall,f1_score,auc,precision,accuracy
0,Attn_Attn_null_False,72.04% ± 1.91%,78.47% ± 1.21%,98.76% ± 0.21%,90.73% ± 1.22%,97.93% ± 0.10%
1,Attn_Attn_null_True,87.65% ± 3.50%,82.63% ± 3.34%,98.83% ± 0.12%,79.43% ± 3.32%,97.00% ± 0.16%
2,Attn_Max_null_False,70.21% ± 1.65%,77.12% ± 1.85%,98.93% ± 0.15%,92.49% ± 1.00%,97.96% ± 0.02%
3,Attn_Max_null_True,86.13% ± 2.29%,80.51% ± 2.66%,98.78% ± 0.32%,78.65% ± 1.92%,96.76% ± 0.19%
4,Attn_Mean_null_False,68.74% ± 0.39%,75.25% ± 0.84%,98.67% ± 0.53%,89.96% ± 2.87%,97.99% ± 0.02%
5,Attn_Mean_null_True,85.53% ± 2.23%,80.35% ± 3.31%,98.76% ± 0.30%,78.32% ± 3.83%,96.66% ± 0.18%
6,Conv_Attn_null_False,54.54% ± 3.24%,56.35% ± 3.13%,97.65% ± 0.32%,67.52% ± 3.80%,96.49% ± 0.11%
7,Conv_Attn_null_True,71.25% ± 6.25%,64.39% ± 6.25%,98.11% ± 0.12%,66.88% ± 5.82%,94.84% ± 0.24%
8,Conv_Max_null_False,48.84% ± 2.92%,50.89% ± 3.14%,97.63% ± 0.27%,68.02% ± 5.50%,96.30% ± 0.28%
9,Conv_Max_null_True,66.60% ± 3.07%,61.11% ± 1.72%,97.63% ± 0.29%,66.82% ± 4.12%,94.22% ± 0.33%


In [23]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics(data.articles, results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,recall,f1_score,auc,precision,accuracy,hamming_loss
0,Attn_Attn_null_False,72.04% ± 1.91%,78.47% ± 1.21%,98.76% ± 0.21%,90.73% ± 1.22%,73.54% ± 1.16%,2.07% ± 0.10%
1,Attn_Attn_null_True,87.65% ± 3.50%,82.63% ± 3.34%,98.83% ± 0.12%,79.43% ± 3.32%,60.57% ± 1.91%,3.00% ± 0.16%
2,Attn_Max_null_False,70.21% ± 1.65%,77.12% ± 1.85%,98.93% ± 0.15%,92.49% ± 1.00%,73.64% ± 0.62%,2.04% ± 0.02%
3,Attn_Max_null_True,86.13% ± 2.29%,80.51% ± 2.66%,98.78% ± 0.32%,78.65% ± 1.92%,57.18% ± 2.56%,3.24% ± 0.19%
4,Attn_Mean_null_False,68.74% ± 0.39%,75.25% ± 0.84%,98.67% ± 0.53%,89.96% ± 2.87%,73.78% ± 1.08%,2.01% ± 0.02%
5,Attn_Mean_null_True,85.53% ± 2.23%,80.35% ± 3.31%,98.76% ± 0.30%,78.32% ± 3.83%,55.68% ± 2.00%,3.34% ± 0.18%
6,Conv_Attn_null_False,54.54% ± 3.24%,56.35% ± 3.13%,97.65% ± 0.32%,67.52% ± 3.80%,57.05% ± 0.27%,3.51% ± 0.11%
7,Conv_Attn_null_True,71.25% ± 6.25%,64.39% ± 6.25%,98.11% ± 0.12%,66.88% ± 5.82%,41.24% ± 2.83%,5.16% ± 0.24%
8,Conv_Max_null_False,48.84% ± 2.92%,50.89% ± 3.14%,97.63% ± 0.27%,68.02% ± 5.50%,54.70% ± 2.29%,3.70% ± 0.28%
9,Conv_Max_null_True,66.60% ± 3.07%,61.11% ± 1.72%,97.63% ± 0.29%,66.82% ± 4.12%,37.90% ± 1.96%,5.78% ± 0.33%


In [24]:
hparams = metrics.apply(lambda x : pd.Series(get_experiment_params(x['expr'])), axis = 1)
metrics = hparams.join(metrics)
metrics['Experiment'] = metrics['expr']
metrics = metrics.drop(columns = ['expr'])
metrics = metrics[[metrics.columns[-1], *metrics.columns[:-1]]]
metrics.to_excel(os.path.join(dir, "hierarchical_teams_metrics.xlsx"), index = False)
metrics

,Experiment,GNN Variant,Readout Function,Pooling Layer,Undersampling,recall,f1_score,auc,precision,accuracy,hamming_loss
0,Attn_Attn_null_False,Attn,Attn,null,False,72.04% ± 1.91%,78.47% ± 1.21%,98.76% ± 0.21%,90.73% ± 1.22%,73.54% ± 1.16%,2.07% ± 0.10%
1,Attn_Attn_null_True,Attn,Attn,null,True,87.65% ± 3.50%,82.63% ± 3.34%,98.83% ± 0.12%,79.43% ± 3.32%,60.57% ± 1.91%,3.00% ± 0.16%
2,Attn_Max_null_False,Attn,Max,null,False,70.21% ± 1.65%,77.12% ± 1.85%,98.93% ± 0.15%,92.49% ± 1.00%,73.64% ± 0.62%,2.04% ± 0.02%
3,Attn_Max_null_True,Attn,Max,null,True,86.13% ± 2.29%,80.51% ± 2.66%,98.78% ± 0.32%,78.65% ± 1.92%,57.18% ± 2.56%,3.24% ± 0.19%
4,Attn_Mean_null_False,Attn,Mean,null,False,68.74% ± 0.39%,75.25% ± 0.84%,98.67% ± 0.53%,89.96% ± 2.87%,73.78% ± 1.08%,2.01% ± 0.02%
5,Attn_Mean_null_True,Attn,Mean,null,True,85.53% ± 2.23%,80.35% ± 3.31%,98.76% ± 0.30%,78.32% ± 3.83%,55.68% ± 2.00%,3.34% ± 0.18%
6,Conv_Attn_null_False,Conv,Attn,null,False,54.54% ± 3.24%,56.35% ± 3.13%,97.65% ± 0.32%,67.52% ± 3.80%,57.05% ± 0.27%,3.51% ± 0.11%
7,Conv_Attn_null_True,Conv,Attn,null,True,71.25% ± 6.25%,64.39% ± 6.25%,98.11% ± 0.12%,66.88% ± 5.82%,41.24% ± 2.83%,5.16% ± 0.24%
8,Conv_Max_null_False,Conv,Max,null,False,48.84% ± 2.92%,50.89% ± 3.14%,97.63% ± 0.27%,68.02% ± 5.50%,54.70% ± 2.29%,3.70% ± 0.28%
9,Conv_Max_null_True,Conv,Max,null,True,66.60% ± 3.07%,61.11% ± 1.72%,97.63% ± 0.29%,66.82% ± 4.12%,37.90% ± 1.96%,5.78% ± 0.33%


- Visualizations

In [25]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [26]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [27]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [28]:
bar_plot(row_wise_metrics_agg, 'accuracy')